# NIG-TimeNet (PyTorch) on Colab: one notebook for every run

Choose **Runtime > Change runtime type > T4 GPU**, then **Runtime > Run all**. Nothing is pasted by hand and no cell is
edited: the only cell with settings is the next one. All logic lives in `src/nig_timenet_torch/colab/` (tests in
`tests/colab/`); the cells below only call it.

1. **Setup**: mount Drive, clean-clone both repos (`rm -rf` first, so a leftover `/content/tf` cannot break it), print
   each commit, install the package WITHOUT touching Colab's CUDA torch, assert and record the device (rule E1).
2. **Data**: export the TF training data (`preprocessing_output_latest.pkl.gz`) as a contract artifact and verify its
   hashes. The export is skipped when a sound artifact made from the same source already exists.
3. **Runner**: queue -> run -> results. The job is `job.json` on branch `colab-queue`; the result bundle is written to
   Drive first, then pushed to branch `colab-results` (`runs/<job_id>/`), and `republish` retries a failed push (rule E2).
   `job.json` schema: `src/nig_timenet_torch/colab/job.py`.

First run only: paste a GitHub token (fine-grained, both repos, Contents read, and read/write on this one) when asked. It
is not shown, is checked against GitHub, and is saved to `TOKEN_PATH` on Drive.

In [ ]:
TORCH_REPO, TORCH_REF = "yuosef772424/nig-timenet-torch", "main"
TF_REPO, TF_REF = "yuosef772424/crypto-signal-prediction", "master"
EXPECT_DEVICE = "cuda"                         # "cpu" only for a deliberate CPU run; a cuda run on a CPU VM fails
TOKEN_PATH = "/content/drive/MyDrive/crypto/secrets/github_token.txt"
WORKDIR = "/content/nig_runner"                # clones' scratch, logs, git credentials helper: never /tmp (rule E2)
TORCH_DIR, TF_DIR = "/content/nig-timenet-torch", "/content/tf"
DRIVE_RUNS = "/content/drive/MyDrive/nig_timenet_runs"
SOURCE = "/content/drive/MyDrive/crypto_model/preprocessed_data/preprocessing_output_latest.pkl.gz"
ARTIFACT = "/content/drive/MyDrive/crypto_model/torch_artifacts/preprocessing_output_1"
ADOPT_EXISTING_ARTIFACT = False                # True: trust a hand-made artifact at ARTIFACT as made from SOURCE

## 1. Setup
The first cell can only be plain code: nothing of the package exists on the VM yet, so it downloads the one
stdlib-only file `colab/bootstrap.py` with the token and runs it. Everything after it is a normal import.

In [ ]:
import getpass, os, urllib.error, urllib.request
from google.colab import drive
drive.mount("/content/drive")
token = open(TOKEN_PATH).read().strip() if os.path.exists(TOKEN_PATH) else getpass.getpass("GitHub token (not shown): ")
url = f"https://api.github.com/repos/{TORCH_REPO}/contents/src/nig_timenet_torch/colab/bootstrap.py?ref={TORCH_REF}"
req = urllib.request.Request(url, headers={"Authorization": "Bearer " + token.strip(), "User-Agent": "nig-colab",
                                           "Accept": "application/vnd.github.raw"})
try:
    boot_source = urllib.request.urlopen(req, timeout=60).read().decode()
except urllib.error.HTTPError as e:
    raise SystemExit(f"GitHub answered HTTP {e.code}: the token is missing, expired or cannot read {TORCH_REPO}. "
                     f"Delete {TOKEN_PATH} and run this cell again.")
boot = {}
exec(compile(boot_source, "bootstrap.py", "exec"), boot)
token = boot["acquire_token"](TOKEN_PATH, TORCH_REPO, lambda: getpass.getpass("GitHub token (not shown): "))

In [ ]:
shas = boot["setup_clones"](token, WORKDIR, {"torch": (TORCH_REPO, TORCH_REF, TORCH_DIR),
                                             "tf": (TF_REPO, TF_REF, TF_DIR)})
install = boot["install_package"](TORCH_DIR)

In [ ]:
from nig_timenet_torch.colab.session import check_environment
setup = check_environment(EXPECT_DEVICE, shas, install, WORKDIR, DRIVE_RUNS)

## 2. Data: TF training data -> contract artifact
Runs `tools/export_torch_artifact.py` of the TF checkout, then checks the result with this project's loader
(`data_io/artifact.py`: sealed coins excluded, max bar time below the seal, every file re-hashed). The record of what the
artifact was made from is `<ARTIFACT>.source.json` next to it.

In [ ]:
from nig_timenet_torch.colab.prepare import prepare_artifact
prep = prepare_artifact(TF_DIR, SOURCE, ARTIFACT, adopt_existing=ADOPT_EXISTING_ARTIFACT)
print(prep["summary"])

## 3. Runner: queue -> run -> results
Reads `job.json` from branch `colab-queue`, refuses a job id that is already on `colab-results` (unless the job sets
`"rerun": true`), checks out the job's pinned `repo_ref`, asserts the job's device and runs the entry under a wall-clock
guard, validates the outputs (including the device the job recorded), writes the bundle to Drive, pushes it to
`colab-results`, and prints a status block.

In [ ]:
from nig_timenet_torch.colab.runner import Runner
runner = Runner(f"https://github.com/{TORCH_REPO}.git", token, WORKDIR, DRIVE_RUNS, runner_sha=shas["torch"])
result = runner.run_all()

### Republish
If the status block says `push: FAILED`, the bundle is already on Drive (`DRIVE_RUNS/<job_id>`). This cell pushes it
again; it needs nothing from the run above except the bundle folder. In a fresh runtime, after the setup cells:
`python -m nig_timenet_torch.colab republish /content/drive/MyDrive/nig_timenet_runs/<job_id>`.

In [ ]:
from nig_timenet_torch.colab.publish import republish
if result["bundle"] and not result["pushed"]:
    republish(result["bundle"], f"https://github.com/{TORCH_REPO}.git", token, WORKDIR)